# Messy University Admissions Database Rescue
#### The Scenario: You act as a university data analyst. The legacy admissions portal crashed during a system migration, and the backup applicant data is a mess. You must clean the "Applicant Records" dataset.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('admissions_messy.csv')
df

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,NaN,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,NaN,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"
...,...,...,...,...,...,...,...
1045,1330.0,diana MAXIMOFF,dmaximoff@edu.org,NaN,AZ,$862.30,10-18-2023
1046,1466.0,STEVE Allen,sallen@edu.org,Journalism,NaN,$389.22,10-20-2023
1047,1121.0,Charlie ROGERS,crogers@stark.com,History,AZ,"$1,746.27","Aug 17, 2023"
1048,1299.0,peter Wayne,pwayne@dailybugle.com,Journalism,KS,$689.05,2023-06-06


## Handling Missing Data: 
#### The dataset is riddled with NaNs. You must use .dropna() to remove rows completely missing an Application ID. You will use .fillna() to replace missing Intended Major and High School State values with "Unknown" that were lost during the export.

In [6]:
df = df.dropna(subset=['Application ID'])

df['Intended Major'] = df['Intended Major'].fillna('Unknown')
df['High School State'] = df['High School State'].fillna('Unknown')

df.head()

,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,Unknown,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,Unknown,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"


## Handling Duplicates: 
#### A server glitch caused applicants to accidentally submit identical applications twice. You must use .duplicated() to find identical application rows and .drop_duplicates(keep='first') to remove the errors and keep only the original submission.

In [7]:
duplicate_count = df.duplicated().sum()
print(f"Found {duplicate_count} duplicate rows.")

df = df.drop_duplicates(keep='first')
df.head()

Found 45 duplicate rows.


,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date
0,1352.0,wanda DOE,wdoe@gmail.com,Math,NV,$297.91,pending
1,1689.0,PETER JONES,pjones@edu.org,Journalism,Unknown,"$1,939.68",08/14/2023
2,1485.0,wanda MAXIMOFF,wmaximoff@yahoo.com,Math,Unknown,$277.90,2023/06/06
3,1388.0,PETER Parker,pparker@yahoo.com,Agriculture,AZ,$370.25,2023-02-24
4,1031.0,bruce rogers,brogers@stark.com,Physics,NV,$53.93,"Oct 16, 2023"


# Data Type Conversions: 
#### Application Fee is imported as a string (e.g., "$150.50"). You must remove the symbols and use pd.to_numeric() to turn it into a float.

In [8]:
df['Application Fee'] = df['Application Fee'].replace('[\$,]', '', regex=True)
df['Application Fee'] = pd.to_numeric(df['Application Fee'])

print(f"Fee Data Type: {df['Application Fee'].dtype}")
df[['Application Fee']].head()

Fee Data Type: float64


,Application Fee
0,297.91
1,1939.68
2,277.90
3,370.25
4,53.93


# Data Type Conversions: 
#### Replace “pending” and “unknown” text entries in the dates with the previous valid datetime (forward filling).
#### Convert Submission Date from messy text into a proper datetime object using pd.to_datetime().
#### Convert the Submission Date format from various mixed formats into a uniform "M/D/Y" format.

In [9]:
df['Submission Date'] = df['Submission Date'].replace(['pending', 'unknown'], np.nan)

df['Submission Date'] = df['Submission Date'].ffill()

df['Submission Date'] = pd.to_datetime(df['Submission Date'], format='mixed')

df['Submission Date'] = df['Submission Date'].dt.strftime('%m/%d/%Y')

print("Dates standardized to M/D/Y format.")
df[['Submission Date']].head()

Dates standardized to M/D/Y format.


,Submission Date
0,NaN
1,08/14/2023
2,06/06/2023
3,02/24/2023
4,10/16/2023


## String Manipulation: 
#### The Applicant Name column has chaotic capitalization and extra spaces. You will use .str.title() and .str.strip() to fix this. You must also extract the email domain (e.g., extracting "yahoo.com" from "student@yahoo.com") using .str.split() to see which email providers are most common.

In [10]:
df['Applicant Name'] = df['Applicant Name'].str.strip().str.title()

df['Email Domain'] = df['Email'].str.split('@').str[1]

print("Final Cleaning Complete. Most common email domains:")
print(df['Email Domain'].value_counts().head())

df.head()

Final Cleaning Complete. Most common email domains:
Email Domain
gmail.com         144
stark.com         137
dailybugle.com    129
yahoo.com         125
outlook.com       124
Name: count, dtype: int64


,Application ID,Applicant Name,Email,Intended Major,High School State,Application Fee,Submission Date,Email Domain
0,1352.0,Wanda Doe,wdoe@gmail.com,Math,NV,297.91,NaN,gmail.com
1,1689.0,Peter Jones,pjones@edu.org,Journalism,Unknown,1939.68,08/14/2023,edu.org
2,1485.0,Wanda Maximoff,wmaximoff@yahoo.com,Math,Unknown,277.90,06/06/2023,yahoo.com
3,1388.0,Peter Parker,pparker@yahoo.com,Agriculture,AZ,370.25,02/24/2023,yahoo.com
4,1031.0,Bruce Rogers,brogers@stark.com,Physics,NV,53.93,10/16/2023,stark.com
